In [0]:
from pyspark.sql import functions as F

# Specify path
CATALOG = "retail_catalog"
BRONZE_SCHEMA = "bronze"
RAW_PATH = "s3://rukks-retail-databricks-project/raw"
SCHEMA_PATH = "s3://rukks-retail-databricks-project/raw/schemas/bronze"
CHECKPOINT_PATH = "s3://rukks-retail-databricks-project/raw/checkpoints/bronze"


In [0]:
# READ CUSTOMERS
customers_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", f"{SCHEMA_PATH}/customers")
    .option("header", "true")
    .load(f"{RAW_PATH}/customers/")
)

In [0]:
# Add our metadata to the customer table
customers_stream = (
    customers_stream
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

In [0]:
# Write the customer data incrementally
(
    customers_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/customers")
    .trigger(availableNow=True)
    .toTable(f"{CATALOG}.{BRONZE_SCHEMA}.customers_raw")
)

In [0]:
# READ PRODUCTS
products_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", f"{SCHEMA_PATH}/products")
    .option("header", "true")
    .load(f"{RAW_PATH}/products/")
)

In [0]:
# Add our metadata to the product table
products_stream = (
    products_stream
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

In [0]:
# Write the product data incrementally
(
    products_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/products")
    .trigger(availableNow=True)
    .toTable(f"{CATALOG}.{BRONZE_SCHEMA}.products_raw")
)

In [0]:
# READ SALES
from pyspark.sql.types import StringType, IntegerType, DoubleType, StructType, StructField

sales_schema = StructType([
    StructField("SaleID", StringType(), True),
    StructField("CustomerID", StringType(), True),
    StructField("ProductID", StringType(), True),
    StructField("StoreID", StringType(), True),
    StructField("SaleDate", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("UnitPrice", DoubleType(), True),
    StructField("Discount", DoubleType(), True),
    StructField("OrderStatus", StringType(), True),
    StructField("PaymentMethod", StringType(), True)
])

In [0]:
sales_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", f"{SCHEMA_PATH}/sales")
    .option("header", "true")
    .schema(sales_schema)
    .load(f"{RAW_PATH}/sales/")
)

In [0]:
# Add our metadata to the sales table
sales_stream = (
    sales_stream
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

In [0]:
sales_stream.printSchema()

root
 |-- SaleID: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- StoreID: string (nullable = true)
 |-- SaleDate: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- OrderStatus: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)



In [0]:
# Write the sales data incrementally
(
    sales_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/sales")
    .trigger(availableNow=True)
    .toTable(f"{CATALOG}.{BRONZE_SCHEMA}.sales_raw")
)

In [0]:
# READ STORES
stores_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", f"{SCHEMA_PATH}/stores")
    .option("header", "true")
    .load(f"{RAW_PATH}/stores/")
)

In [0]:
# Add our metadata to the stores table
stores_stream = (
    stores_stream
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

In [0]:
# Write the stores data incrementally
(
    stores_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/stores")
    .trigger(availableNow=True)
    .toTable(f"{CATALOG}.{BRONZE_SCHEMA}.stores_raw")
)